In [ ]:
%load_ext autoreload
%autoreload 2

# MH-Weed16: Real-Time Multiclass Weed Detection & Analytics with YOLO11

A complete end-to-end computer vision and exploratory analytics pipeline for agricultural weed detection using the MH-Weed16 benchmark dataset and Ultralytics YOLO11 Nano.

### Pipeline Architecture
- Offline downsampling to 640x360 with exact aspect ratio preservation.
- Exploratory data analysis (class frequency, spatial heatmaps, co-occurrence matrices, bounding box geometry).
- Multi-modal feature engineering (geometric box morphology, RGB/HSV color statistics, ExG/ExR/NDI vegetation indices, texture/Laplacian sharpness).
- Principal Component Analysis (PCA) for variance explanation, 2D weed manifold visualization, and feature loadings.
- Diagnostic outlier detection via statistical IQR rules and Isolation Forest anomaly scoring.
- Bounding box anchor geometry analysis via K-Means clustering.
- Agricultural foliage contrast enhancement (CLAHE and Excess Green Index).
- Exhaustive genetic hyperparameter optimization via Ultralytics model.tune.
- High-throughput end-to-end YOLO11 Nano training with RAM caching and tuned hyperparameters.
- Comprehensive test set validation (mAP50, mAP50-95, Precision, Recall).
- Side-by-side ground truth vs prediction inference visualization.
- Dynamic ONNX export for edge deployment on agricultural sprayers and robots.

## 1. Setup & Environment
Initialize libraries, configure deterministic random seeds, and detect compute hardware acceleration (Intel Arc XPU, NVIDIA CUDA, or CPU fallback).

In [ ]:
import concurrent.futures
import os
import pathlib
import random
import subprocess
import time
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import torch
from ultralytics import YOLO
import yaml

# Deterministic seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Detect compute device
if hasattr(torch, 'xpu') and torch.xpu.is_available():
    device = 'xpu'
    device_name = torch.xpu.get_device_name(0)
elif torch.cuda.is_available():
    device = '0'
    device_name = torch.cuda.get_device_name(0)
else:
    device = 'cpu'
    device_name = 'CPU'

print("=" * 65)
print(f"PyTorch Version:  {torch.__version__}")
print(f"Compute Device:   {device} ({device_name})")
print("=" * 65)

ROOT_DIR = pathlib.Path(".").resolve()

## 2. Dataset Preparation & Offline Downsampling (640x360)

### Why Was Training Slow Previously?
The raw MH-Weed16 dataset contains 6,656 Full HD (1920x1080) JPEG images. When training end-to-end with high-resolution images, the CPU dataloader spends over 90% of each epoch uncompressing, decoding, and scaling 2-megapixel JPEGs on the fly. This starves the GPU/XPU, keeping compute cores idle.

### Offline Downsampling with Exact Aspect Ratio Preservation
1. Aspect Ratio Preservation: Downsampling images to 640x360 (exact 1/3 scale of 1920x1080).
2. Dimensionless Bounding Boxes: In YOLO format, annotations (x, y, w, h) are normalized relative to image width and height:
   $$\frac{x}{1920} = \frac{x / 3}{640}, \quad \frac{y}{1080} = \frac{y / 3}{360}$$
   Because the aspect ratio is strictly preserved, normalized bounding box coordinates remain 100% mathematically identical.
3. Pristine Raw Dataset: The original MH-Weed16 directory is never modified. Resized images are cached in data/resized/images, and labels are accessed via direct symlink/junction in data/resized/labels.
4. RAM Caching: At 640x360, the entire dataset footprint is reduced from ~6 GB to ~260 MB, fitting completely inside RAM (cache=True) for zero-disk-latency epochs.

In [ ]:
DATA_DIR = ROOT_DIR / "data" / "resized"
IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"

IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Locate raw dataset paths
RAW_IMAGES = ROOT_DIR / "MH-Weed16" / "Crop with Weeds" / "intel Real Sense Depth_Clicks" / "intel Real Sense Depth_Clicks"
RAW_LABELS = ROOT_DIR / "MH-Weed16" / "Crop with Weeds" / "intel Real Sense Depth_Annotations" / "intel Real Sense Depth_Annotations" / "YOLO_darknet"

if not RAW_IMAGES.exists():
    raise FileNotFoundError(f"Raw images directory not found at: {RAW_IMAGES}")
if not RAW_LABELS.exists():
    raise FileNotFoundError(f"Raw labels directory not found at: {RAW_LABELS}")

# 1. Create junction/symlink for labels to point directly to RAW_LABELS
if not LABELS_DIR.exists():
    if os.name == 'nt':
        cmd = f'cmd /c mklink /J "{LABELS_DIR}" "{RAW_LABELS}"'
        subprocess.run(cmd, shell=True, check=True)
    else:
        LABELS_DIR.symlink_to(RAW_LABELS)
    print(f"Linked labels to raw annotations: {LABELS_DIR}")
else:
    print(f"Labels link already exists: {LABELS_DIR}")

# 2. Downsample images to 640x360 in parallel
raw_image_files = sorted([p for p in RAW_IMAGES.glob('*.*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')])
print(f"Raw images discovered: {len(raw_image_files):,}")

TARGET_WIDTH = 640
TARGET_HEIGHT = 360

def resize_task(img_path):
    out_path = IMAGES_DIR / img_path.name
    if out_path.exists() and out_path.stat().st_size > 0:
        return False
    img = cv2.imread(str(img_path))
    if img is None:
        return False
    resized = cv2.resize(img, (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_AREA)
    cv2.imwrite(str(out_path), resized, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
    return True

pending_images = [p for p in raw_image_files if not (IMAGES_DIR / p.name).exists()]
if pending_images:
    print(f"Downsampling {len(pending_images):,} images to {TARGET_WIDTH}x{TARGET_HEIGHT} (using multithreading)...")
    t0 = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
        list(executor.map(resize_task, pending_images))
    t1 = time.time()
    print(f"Finished downsampling in {t1 - t0:.1f}s ({(t1 - t0)/len(pending_images)*1000:.1f}ms/img)")
else:
    print(f"All {len(raw_image_files):,} images are already downsampled in {IMAGES_DIR}.")

# 3. Create 80 / 10 / 10 Train / Val / Test Splits
resized_image_files = sorted([p for p in IMAGES_DIR.glob('*.*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')])
all_image_paths = [os.path.normpath(str(p)) for p in resized_image_files]

random.seed(SEED)
shuffled_paths = list(all_image_paths)
random.shuffle(shuffled_paths)

n_total = len(shuffled_paths)
n_train = int(0.80 * n_total)
n_val = int(0.10 * n_total)

train_paths = shuffled_paths[:n_train]
val_paths = shuffled_paths[n_train:n_train + n_val]
test_paths = shuffled_paths[n_train + n_val:]

def write_split_file(filepath, paths):
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write('\n'.join(paths) + '\n')

write_split_file(ROOT_DIR / "train.txt", train_paths)
write_split_file(ROOT_DIR / "val.txt", val_paths)
write_split_file(ROOT_DIR / "test.txt", test_paths)

print(f"Splits saved: Train={len(train_paths):,} | Val={len(val_paths):,} | Test={len(test_paths):,}")

# 4. Species mapping and dataset YAML configuration
CLASS_NAMES = {
    0: "kena",
    1: "lavhala",
    2: "lambs_quarters",
    3: "little_mallow",
    4: "moti_dudhi",
    5: "obscure_morning_glory",
    6: "asian_pigeonwings",
    7: "bilayat",
    8: "choti_dudhi",
    9: "digitaria",
    10: "gajar_gavat",
    11: "graceful_sandmat",
    12: "sicklepod",
    13: "harali",
    14: "dwarf_cassia"
}

yaml_content = {
    'path': str(ROOT_DIR).replace('\\', '/'),
    'train': 'train.txt',
    'val': 'val.txt',
    'test': 'test.txt',
    'names': CLASS_NAMES
}

yaml_file = ROOT_DIR / "mhweed16.yaml"
with open(yaml_file, 'w', encoding='utf-8') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)

print(f"Updated dataset YAML: {yaml_file}")

## 3. Exploratory Data Analysis (EDA) & Dataset Topology
Examine species representation, instance densities, co-occurrence behavior across field images, and spatial bounding box placement.

In [ ]:
ann_files = sorted(list(LABELS_DIR.glob('*.txt')))
box_records = []
class_counts = Counter()
boxes_per_image = []
image_class_sets = {}

for ann_path in ann_files:
    with open(ann_path, 'r', encoding='utf-8') as f:
        lines = [line.strip().split() for line in f if line.strip()]
    
    boxes_per_image.append(len(lines))
    present_classes = set()
    
    for parts in lines:
        cls_id = int(parts[0])
        xc, yc, w, h = map(float, parts[1:5])
        class_counts[cls_id] += 1
        present_classes.add(cls_id)
        box_records.append({
            'image_stem': ann_path.stem,
            'class_id': cls_id,
            'name': CLASS_NAMES.get(cls_id, str(cls_id)),
            'xc': xc,
            'yc': yc,
            'width': w,
            'height': h,
            'aspect_ratio': w / max(h, 1e-6),
            'area': w * h,
            'perimeter': 2.0 * (w + h),
            'center_dist': np.sqrt((xc - 0.5)**2 + (yc - 0.5)**2)
        })
    image_class_sets[ann_path.stem] = present_classes

df_boxes = pd.DataFrame(box_records)
print(f"Total Bounding Box Instances: {len(df_boxes):,}")
print(f"Total Labeled Images:         {len(ann_files):,}")
print(f"Unique Weed Species Present:  {len(class_counts)}")
print(f"Mean Boxes Per Image:         {np.mean(boxes_per_image):.2f}")
print(f"Max Boxes in a Single Image:  {np.max(boxes_per_image)}")

# 1. Species Distribution & Boxes Per Image
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

order = [CLASS_NAMES[i] for i in range(len(CLASS_NAMES)) if i in class_counts]
sns.countplot(data=df_boxes, x='name', order=order, ax=axes[0], color='#2c7bb6')
axes[0].set_title("Instance Frequency per Weed Species", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Weed Species")
axes[0].set_ylabel("Instance Count")
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

sns.histplot(boxes_per_image, bins=range(0, 30), ax=axes[1], color='#1a9641', kde=False)
axes[1].set_title("Bounding Box Density per Image Frame", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Number of Weed Bounding Boxes")
axes[1].set_ylabel("Image Count")
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# 2. Spatial Heatmap (Field-of-View Distribution) & Weed Co-Occurrence Matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Spatial Heatmap of Box Centers across camera frame
sns.kdeplot(
    data=df_boxes,
    x='xc',
    y='yc',
    cmap='YlOrRd',
    fill=True,
    thresh=0.02,
    levels=15,
    ax=axes[0]
)
axes[0].scatter(
    df_boxes['xc'].sample(min(1500, len(df_boxes)), random_state=SEED),
    df_boxes['yc'].sample(min(1500, len(df_boxes)), random_state=SEED),
    s=2,
    alpha=0.2,
    color='#252525'
)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(1, 0)  # Invert Y to match image coordinates
axes[0].set_title("Spatial Density of Weed Centroids (Normalized Frame Coordinates)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Normalized X Center (xc)")
axes[0].set_ylabel("Normalized Y Center (yc)")

# Weed Co-occurrence Matrix
all_cids = sorted(list(CLASS_NAMES.keys()))
co_matrix = np.zeros((len(all_cids), len(all_cids)), dtype=int)

for cls_set in image_class_sets.values():
    for c1 in cls_set:
        for c2 in cls_set:
            co_matrix[c1, c2] += 1

labels = [CLASS_NAMES[i] for i in all_cids]
sns.heatmap(
    co_matrix,
    xticklabels=labels,
    yticklabels=labels,
    cmap='Blues',
    annot=False,
    ax=axes[1],
    cbar=True
)
axes[1].set_title("Weed Species Co-Occurrence Matrix in Camera Frames", fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# 3. Bounding Box Geometric Distributions
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

sns.histplot(df_boxes['width'], bins=35, kde=True, ax=axes[0], color='#2c7bb6')
axes[0].set_title("Normalized Box Width")
axes[0].set_xlabel("Width (relative)")

sns.histplot(df_boxes['height'], bins=35, kde=True, ax=axes[1], color='#1a9641')
axes[1].set_title("Normalized Box Height")
axes[1].set_xlabel("Height (relative)")

sns.histplot(df_boxes['aspect_ratio'], bins=35, kde=True, ax=axes[2], color='#fdae61')
axes[2].set_title("Aspect Ratio (Width / Height)")
axes[2].set_xlabel("Aspect Ratio")
axes[2].set_xlim(0, 5)

sns.histplot(df_boxes['area'], bins=35, kde=True, ax=axes[3], color='#d7191c')
axes[3].set_title("Normalized Box Area")
axes[3].set_xlabel("Area (relative)")

plt.tight_layout()
plt.show()

In [ ]:
# 4. Ground Truth Bounding Box Alignment on Sample Resized Image
def show_sample_ground_truth(img_path, ann_path):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    if ann_path.exists():
        with open(ann_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
                
                x1 = int((xc - bw / 2) * w)
                y1 = int((yc - bh / 2) * h)
                x2 = int((xc + bw / 2) * w)
                y2 = int((yc + bh / 2) * h)
                
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                name = CLASS_NAMES.get(cls_id, str(cls_id))
                cv2.putText(img, name, (x1, max(18, y1 - 4)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)
    
    plt.figure(figsize=(10, 5.5))
    plt.imshow(img)
    plt.title(f"Resized Sample ({w}x{h}): {img_path.name} | Ground Truth Bounding Boxes", fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

sample_img = resized_image_files[0]
sample_ann = LABELS_DIR / f"{sample_img.stem}.txt"
show_sample_ground_truth(sample_img, sample_ann)

## 4. Feature Engineering: Multi-Modal Bounding Box & Patch Descriptors
Extract comprehensive quantitative descriptors across every weed instance:
- Geometric features: normalized width, height, aspect ratio (w/h), bounding box area, perimeter, and radial distance from image center.
- Color & Spectral features: channel means and standard deviations in RGB and HSV color spaces.
- Agricultural Vegetation Indices: Excess Green Index ($\\text{ExG} = 2G - R - B$), Excess Red ($\\text{ExR} = 1.4R - G$), and Normalized Difference Index ($\\text{NDI} = (G - R) / (G + R + 1e-6)$).
- Texture & Gradient metrics: Grayscale intensity standard deviation (contrast) and Laplacian variance (edge sharpness and foliage texture).

Computed entirely in memory across the downsampled image set using parallel multi-threading.

In [ ]:
def extract_instance_features(img_path):
    ann_path = LABELS_DIR / f"{img_path.stem}.txt"
    if not ann_path.exists():
        return []
    
    img = cv2.imread(str(img_path))
    if img is None:
        return []
    
    h_img, w_img = img.shape[:2]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    with open(ann_path, 'r', encoding='utf-8') as f:
        lines = [line.strip().split() for line in f if line.strip()]
    
    n_boxes = len(lines)
    records = []
    
    for parts in lines:
        cls_id = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:5])
        
        x1 = max(0, int((xc - bw / 2.0) * w_img))
        y1 = max(0, int((yc - bh / 2.0) * h_img))
        x2 = min(w_img, int((xc + bw / 2.0) * w_img))
        y2 = min(h_img, int((yc + bh / 2.0) * h_img))
        
        patch_rgb = img_rgb[y1:y2, x1:x2]
        if patch_rgb.size < 12:
            continue
        
        patch_hsv = img_hsv[y1:y2, x1:x2]
        patch_gray = img_gray[y1:y2, x1:x2]
        
        # Normalize RGB to [0, 1]
        rf = patch_rgb[:, :, 0].astype(np.float32) / 255.0
        gf = patch_rgb[:, :, 1].astype(np.float32) / 255.0
        bf = patch_rgb[:, :, 2].astype(np.float32) / 255.0
        
        exg = 2.0 * gf - rf - bf
        exr = 1.4 * rf - gf
        denom = gf + rf + 1e-6
        ndi = (gf - rf) / denom
        
        lap_var = float(cv2.Laplacian(patch_gray, cv2.CV_64F).var()) if patch_gray.size >= 16 else 0.0
        
        records.append({
            'image_name': img_path.name,
            'class_id': cls_id,
            'species': CLASS_NAMES.get(cls_id, str(cls_id)),
            'xc': xc,
            'yc': yc,
            'width': bw,
            'height': bh,
            'aspect_ratio': bw / max(bh, 1e-6),
            'area': bw * bh,
            'perimeter': 2.0 * (bw + bh),
            'center_dist': np.sqrt((xc - 0.5)**2 + (yc - 0.5)**2),
            'boxes_in_frame': n_boxes,
            'r_mean': float(np.mean(rf)),
            'g_mean': float(np.mean(gf)),
            'b_mean': float(np.mean(bf)),
            'r_std': float(np.std(rf)),
            'g_std': float(np.std(gf)),
            'b_std': float(np.std(bf)),
            'hue_mean': float(np.mean(patch_hsv[:, :, 0])),
            'sat_mean': float(np.mean(patch_hsv[:, :, 1])),
            'val_mean': float(np.mean(patch_hsv[:, :, 2])),
            'exg_mean': float(np.mean(exg)),
            'exg_std': float(np.std(exg)),
            'exr_mean': float(np.mean(exr)),
            'ndi_mean': float(np.mean(ndi)),
            'contrast_std': float(np.std(patch_gray)),
            'laplacian_var': lap_var
        })
        
    return records

print(f"Extracting multi-modal features across {len(resized_image_files):,} downsampled images...")
t0 = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    extracted_batches = list(executor.map(extract_instance_features, resized_image_files))

feature_list = [r for batch in extracted_batches for r in batch]
df_features = pd.DataFrame(feature_list)
t1 = time.time()

print(f"Extraction complete in {t1 - t0:.2f}s: Extracted {len(df_features):,} weed instance records with {df_features.shape[1]} features.")
df_features.head()

In [ ]:
# Multi-Modal Feature Analysis by Weed Species
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Excess Green Index (ExG) distribution across species
species_order = df_features['species'].value_counts().index[:10]  # Top 10 species by count
sns.boxplot(
    data=df_features[df_features['species'].isin(species_order)],
    x='species',
    y='exg_mean',
    order=species_order,
    ax=axes[0],
    palette='Greens_r'
)
axes[0].set_title("Excess Green Index (ExG) Distribution by Species", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Weed Species")
axes[0].set_ylabel("Mean ExG Index")
axes[0].tick_params(axis='x', rotation=40)

# 2. Aspect Ratio vs Normalized Area (Log Scale)
sns.scatterplot(
    data=df_features.sample(min(3000, len(df_features)), random_state=SEED),
    x='aspect_ratio',
    y='area',
    hue='species',
    alpha=0.6,
    s=15,
    legend=False,
    ax=axes[1]
)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_title("Aspect Ratio vs Box Area (Log-Log Scale)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Aspect Ratio (log scale)")
axes[1].set_ylabel("Box Area (log scale)")

# 3. Foliage Edge Sharpness (Laplacian Variance)
sns.violinplot(
    data=df_features[df_features['species'].isin(species_order)],
    x='species',
    y='laplacian_var',
    order=species_order,
    ax=axes[2],
    cut=0,
    palette='Blues_r'
)
axes[2].set_ylim(0, np.percentile(df_features['laplacian_var'], 98))
axes[2].set_title("Foliage Edge Sharpness (Laplacian Variance)", fontsize=12, fontweight='bold')
axes[2].set_xlabel("Weed Species")
axes[2].set_ylabel("Laplacian Variance")
axes[2].tick_params(axis='x', rotation=40)

plt.tight_layout()
plt.show()

## 5. Principal Component Analysis (PCA) & Dimensionality Reduction
Standardize the multi-modal feature matrix and project high-dimensional descriptors into principal orthogonal axes to evaluate variance contribution, inspect cluster separability, and identify key driving features.

In [ ]:
feature_cols = [
    'width', 'height', 'aspect_ratio', 'area', 'perimeter', 'center_dist', 'boxes_in_frame',
    'r_mean', 'g_mean', 'b_mean', 'r_std', 'g_std', 'b_std',
    'hue_mean', 'sat_mean', 'val_mean', 'exg_mean', 'exg_std', 'exr_mean', 'ndi_mean',
    'contrast_std', 'laplacian_var'
]

X = df_features[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=10, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

for i in range(1, 5):
    df_features[f'PC{i}'] = X_pca[:, i - 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Scree Plot (Explained Variance Ratio)
evr = pca.explained_variance_ratio_
cum_evr = np.cumsum(evr)
axes[0].bar(range(1, 11), evr, alpha=0.6, color='#2c7bb6', label='Individual Variance', align='center')
axes[0].step(range(1, 11), cum_evr, where='mid', color='#d7191c', linewidth=2, label='Cumulative Variance')
axes[0].set_title("PCA Scree Plot (Explained Variance)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Explained Variance Ratio")
axes[0].set_xticks(range(1, 11))
axes[0].grid(axis='y', linestyle='--', alpha=0.5)
axes[0].legend(loc='best')

# 2. 2D PCA Projection (PC1 vs PC2)
top_species = df_features['species'].value_counts().index[:6]
df_pca_sub = df_features[df_features['species'].isin(top_species)]
sns.scatterplot(
    data=df_pca_sub.sample(min(2500, len(df_pca_sub)), random_state=SEED),
    x='PC1',
    y='PC2',
    hue='species',
    alpha=0.5,
    s=18,
    ax=axes[1]
)
axes[1].set_title("2D PCA Projection (PC1 vs PC2)", fontsize=12, fontweight='bold')
axes[1].set_xlabel(f"PC1 ({evr[0]*100:.1f}% variance)")
axes[1].set_ylabel(f"PC2 ({evr[1]*100:.1f}% variance)")
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Species')

# 3. PCA Feature Loadings Heatmap for PC1 and PC2
loadings = pd.DataFrame(
    pca.components_[:2, :].T,
    index=feature_cols,
    columns=['PC1 Loading', 'PC2 Loading']
)
sns.heatmap(loadings, annot=True, cmap='vlag', fmt='.2f', ax=axes[2], cbar=True)
axes[2].set_title("Feature Loadings for PC1 & PC2", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Outlier Detection & Anomaly Auditing
Auditing dataset quality without discarding standard benchmark ground truth annotations:
- Statistical Geometry Outliers: Bounding boxes with abnormal aspect ratios (w/h > 8 or < 0.125), degenerate micro-boxes (width or height < 0.01), or oversized boxes (area > 0.40).
- Multi-Modal Anomaly Scoring: Isolation Forest (contamination=0.01) fitted to the standardized multi-modal feature matrix.
- Visual Audit: Inspecting high-anomaly bounding box instances to diagnose edge clipping, motion blur, or severe lighting variation.

In [ ]:
# 1. Statistical Rules
df_features['outlier_geom'] = (
    (df_features['aspect_ratio'] > 8.0) |
    (df_features['aspect_ratio'] < 0.125) |
    (df_features['width'] < 0.01) |
    (df_features['height'] < 0.01) |
    (df_features['area'] > 0.40)
)

# 2. Isolation Forest Multi-Modal Anomaly Detection
iso_forest = IsolationForest(n_estimators=100, contamination=0.01, random_state=SEED, n_jobs=-1)
iso_forest.fit(X_scaled)

df_features['anomaly_score'] = -iso_forest.score_samples(X_scaled)
df_features['outlier_iso'] = iso_forest.predict(X_scaled) == -1

print("=" * 60)
print("             DATASET OUTLIER AUDIT SUMMARY            ")
print("=" * 60)
print(f"Total Weed Instances Analyzed: {len(df_features):,}")
print(f"Geometric Rule Outliers:       {df_features['outlier_geom'].sum():,} ({df_features['outlier_geom'].mean()*100:.2f}%)")
print(f"Isolation Forest Outliers:     {df_features['outlier_iso'].sum():,} ({df_features['outlier_iso'].mean()*100:.2f}%)")
print(f"Overlap (Both Flags):          {(df_features['outlier_geom'] & df_features['outlier_iso']).sum():,}")
print("=" * 60)

plt.figure(figsize=(10, 4))
sns.histplot(df_features['anomaly_score'], bins=50, kde=True, color='#d7191c', label='Anomaly Score')
thresh = np.percentile(df_features['anomaly_score'], 99)
plt.axvline(thresh, color='black', linestyle='--', label=f'99th Percentile Threshold ({thresh:.3f})')
plt.title("Isolation Forest Anomaly Score Distribution", fontsize=12, fontweight='bold')
plt.xlabel("Anomaly Score (higher = more anomalous)")
plt.ylabel("Instance Count")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Visual Audit of Top Detected Anomaly Instances
top_anomalies = df_features.sort_values(by='anomaly_score', ascending=False).head(6)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for idx, (_, row) in enumerate(top_anomalies.iterrows()):
    img_p = IMAGES_DIR / row['image_name']
    if not img_p.exists():
        continue
    raw = cv2.imread(str(img_p))
    h_img, w_img = raw.shape[:2]
    
    x1 = max(0, int((row['xc'] - row['width'] / 2.0) * w_img))
    y1 = max(0, int((row['yc'] - row['height'] / 2.0) * h_img))
    x2 = min(w_img, int((row['xc'] + row['width'] / 2.0) * w_img))
    y2 = min(h_img, int((row['yc'] + row['height'] / 2.0) * h_img))
    
    # Crop patch with context padding
    pad_x = int(0.2 * (x2 - x1))
    pad_y = int(0.2 * (y2 - y1))
    crop_x1 = max(0, x1 - pad_x)
    crop_y1 = max(0, y1 - pad_y)
    crop_x2 = min(w_img, x2 + pad_x)
    crop_y2 = min(h_img, y2 + pad_y)
    
    patch = cv2.cvtColor(raw[crop_y1:crop_y2, crop_x1:crop_x2], cv2.COLOR_BGR2RGB)
    axes[idx].imshow(patch)
    axes[idx].set_title(f"Species: {row['species']}\nScore: {row['anomaly_score']:.3f} | AR: {row['aspect_ratio']:.2f}", fontsize=10)
    axes[idx].axis('off')

plt.suptitle("Visual Audit of Top Multi-Modal Anomaly Instances", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Bounding Box Anchor Geometry Analysis (K-Means)
Cluster normalized bounding box widths and heights using K-Means to identify dominant spatial aspect ratios and scales in MH-Weed16.

In [ ]:
wh_data = df_features[['width', 'height']].values
k_clusters = 6
kmeans = KMeans(n_clusters=k_clusters, random_state=SEED, n_init=10)
cluster_labels = kmeans.fit_predict(wh_data)
centroids = kmeans.cluster_centers_

plt.figure(figsize=(9, 6))
plt.scatter(
    df_features['width'].sample(min(3000, len(df_features)), random_state=SEED),
    df_features['height'].sample(min(3000, len(df_features)), random_state=SEED),
    c=cluster_labels[:min(3000, len(df_features))],
    cmap='tab10',
    s=8,
    alpha=0.3,
    label='Weed Instances'
)
plt.scatter(
    centroids[:, 0],
    centroids[:, 1],
    c='black',
    s=120,
    marker='X',
    edgecolor='white',
    linewidth=1.5,
    label='Cluster Centroids (k=6)'
)

for i, (cw, ch) in enumerate(centroids):
    plt.text(
        cw + 0.005,
        ch + 0.005,
        f"C{i+1}: ({cw:.2f}, {ch:.2f})\nAR={cw/max(ch, 1e-5):.2f}",
        fontsize=9,
        fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='yellow', alpha=0.6)
    )

plt.title("K-Means Clustering of Bounding Box Dimensions (Width vs Height)", fontsize=12, fontweight='bold')
plt.xlabel("Normalized Width")
plt.ylabel("Normalized Height")
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 8. Agricultural Preprocessing & Foliage Contrast (CLAHE & ExG)
Field imaging in agriculture is subject to harsh ambient lighting, specular leaf reflection, and varying soil moisture. Below we illustrate Contrast-Limited Adaptive Histogram Equalization (CLAHE) and the Excess Green Index ($\\text{ExG} = 2G - R - B$) used to segment photosynthetic vegetation.

In [ ]:
def apply_clahe(img_rgb, clip_limit=2.0):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
    l_eq = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((l_eq, a, b)), cv2.COLOR_LAB2RGB)

def compute_exg(img_rgb):
    f = img_rgb.astype(np.float32) / 255.0
    r, g, b = f[:, :, 0], f[:, :, 1], f[:, :, 2]
    return 2.0 * g - r - b

raw_sample = cv2.cvtColor(cv2.imread(str(resized_image_files[0])), cv2.COLOR_BGR2RGB)
clahe_sample = apply_clahe(raw_sample)
exg_sample = compute_exg(raw_sample)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(raw_sample)
axes[0].set_title("Resized RGB (640x360)")
axes[0].axis('off')

axes[1].imshow(clahe_sample)
axes[1].set_title("CLAHE Illumination Equalized")
axes[1].axis('off')

im3 = axes[2].imshow(exg_sample, cmap='YlGn')
axes[2].set_title("Excess Green Index (ExG)")
axes[2].axis('off')
plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 9. Hyperparameter Tuning via model.tune (Exhaustive Genetic Evolution)
Ultralytics YOLO provides genetic hyperparameter evolution via `model.tune()`. Over successive iterations, candidate hyperparameter combinations are evaluated, mutating top configurations based on validation fitness:

$$\\text{Fitness} = 0.1 \\times \\text{mAP@0.50} + 0.9 \\times \\text{mAP@0.50:0.95}$$

### Tuning Budget Configuration
- Iterations: 30 generations of genetic mutation.
- Epochs per iteration: 20 epochs per trial.
- Tuned parameters: Learning rate schedule (`lr0`, `lrf`), SGD/AdamW momentum, weight decay, loss gains (`box`, `cls`, `dfl`), and spatial augmentations (`degrees`, `translate`, `scale`, `shear`, `flipud`, `fliplr`, `mosaic`, `mixup`, `copy_paste`).
- Destination: Checkpoints and `best_hyperparameters.yaml` are written to `runs/detect/tune`.
- Execution Control: Toggle `RUN_TUNE = True` to launch the overnight search. If already executed or skipped, the notebook automatically loads `best_hyperparameters.yaml` if available, or falls back cleanly to optimized parameters.

In [ ]:
RUN_TUNE = False  # Set to True to launch exhaustive overnight hyperparameter tuning run

tune_dir = ROOT_DIR / "runs" / "detect" / "tune"
best_hyp_yaml = tune_dir / "best_hyperparameters.yaml"

tune_iterations = 30
tune_epochs = 20

if RUN_TUNE:
    print("=" * 65)
    print("Launching Exhaustive Genetic Hyperparameter Tuning (model.tune)...")
    print(f"Budget: {tune_iterations} iterations at {tune_epochs} epochs per trial")
    print(f"Device: {device} ({device_name}) | Image Size: 640 | Batch: 16")
    print("=" * 65)
    
    tune_model = YOLO("yolo11n.pt")
    tune_results = tune_model.tune(
        data="mhweed16.yaml",
        epochs=tune_epochs,
        iterations=tune_iterations,
        batch=16,
        imgsz=640,
        device=device,
        workers=4,
        cache=True,
        amp=False,  # FP32 for Intel Arc XPU stability
        project=str(ROOT_DIR / "runs" / "detect"),
        name="tune",
        exist_ok=True
    )
    print("Hyperparameter evolution complete.")
else:
    print("RUN_TUNE is set to False (overnight run bypassed).")

# Check if best_hyperparameters.yaml exists from a prior or current run
if best_hyp_yaml.exists():
    print(f"Discovered evolved hyperparameters at: {best_hyp_yaml}")
    with open(best_hyp_yaml, 'r', encoding='utf-8') as f:
        best_hyperparameters = yaml.safe_load(f)
    print("\nEvolved Hyperparameters Configuration:")
    for k, v in best_hyperparameters.items():
        print(f"  {k:15s}: {v}")
else:
    print(f"No prior tuning output found at {best_hyp_yaml}. Using optimized agricultural baseline hyperparameters.")
    best_hyperparameters = {
        'lr0': 0.01,
        'lrf': 0.01,
        'momentum': 0.937,
        'weight_decay': 0.0005,
        'warmup_epochs': 3.0,
        'box': 7.5,
        'cls': 0.5,
        'dfl': 1.5,
        'degrees': 45.0,
        'flipud': 0.5,
        'fliplr': 0.5,
        'mosaic': 1.0,
        'mixup': 0.1
    }

## 10. Model Training (YOLO11 Nano End-to-End)
Train the detector end-to-end with Ultralytics YOLO11 Nano (`yolo11n.pt`), injecting evolved hyperparameters dynamically.

### Optimized Training Configuration
- Batch Size 16: High throughput.
- Workers 4: Parallel PyTorch dataloader threads to feed batches instantaneously.
- RAM Caching (`cache=True`): Pre-loads the downsampled dataset into system RAM for zero disk latency.
- Target Size `imgsz=640`: Matches model receptive field.
- Precision: `amp=False` is set for stability on Intel Arc XPU.
- Fault-Tolerant Resumption: Automatically detects `last.pt` and resumes if interrupted.

In [ ]:
epochs = 50
batch = 16
imgsz = 640
workers = 4
run_name = "mhweed16_yolo11n"

run_dir = ROOT_DIR / "runs" / "detect" / run_name
last_weights = run_dir / "weights" / "last.pt"
best_weights = run_dir / "weights" / "best.pt"

print("=" * 65)
print(f"Run Name:      {run_name}")
print(f"Device:        {device} ({device_name})")
print(f"Epochs:        {epochs} | Batch: {batch} | Image Size: {imgsz} | Workers: {workers}")
print(f"RAM Caching:   Enabled (cache=True)")
print("=" * 65)

if last_weights.exists():
    print(f"Resuming training from checkpoint: {last_weights}")
    model = YOLO(str(last_weights))
    results = model.train(resume=True)
else:
    print("Starting fresh end-to-end training with yolo11n.pt...")
    model = YOLO("yolo11n.pt")
    
    train_args = {
        'data': "mhweed16.yaml",
        'epochs': epochs,
        'batch': batch,
        'imgsz': imgsz,
        'device': device,
        'workers': workers,
        'cache': True,
        'name': run_name,
        'exist_ok': True,
        'amp': False,
        'patience': 15
    }
    
    # Merge tuned hyperparameters
    for hyp_key, hyp_val in best_hyperparameters.items():
        if hyp_key in ['lr0', 'lrf', 'momentum', 'weight_decay', 'warmup_epochs',
                        'box', 'cls', 'dfl', 'degrees', 'translate', 'scale', 'shear',
                        'flipud', 'fliplr', 'mosaic', 'mixup', 'copy_paste']:
            train_args[hyp_key] = hyp_val
            
    results = model.train(**train_args)

print(f"Training complete. Best checkpoint saved at: {best_weights if best_weights.exists() else 'N/A'}")

## 11. Evaluation & Test Set Metrics
Evaluate model performance on the held-out test split (`test.txt`). Computes mAP50, mAP50-95, precision, and recall.

In [ ]:
eval_weights = best_weights if best_weights.exists() else pathlib.Path("yolo11n.pt")
print(f"Evaluating checkpoint: {eval_weights}")
eval_model = YOLO(str(eval_weights))

metrics = eval_model.val(
    data="mhweed16.yaml",
    split="test",
    device=device,
    batch=batch,
    imgsz=imgsz
)

print("\n" + "=" * 45)
print("           TEST SET EVALUATION METRICS       ")
print("=" * 45)
print(f"  mAP@0.50:       {metrics.box.map50:.4f}")
print(f"  mAP@0.50:0.95:  {metrics.box.map:.4f}")
print(f"  Mean Precision: {metrics.box.mp:.4f}")
print(f"  Mean Recall:    {metrics.box.mr:.4f}")
print("=" * 45)

## 12. Inference Visualization & Comparison
Generate visual detections on test set images and compare model predictions against ground truth bounding boxes side-by-side.

In [ ]:
with open("test.txt", 'r', encoding='utf-8') as f:
    test_image_paths = [pathlib.Path(line.strip()) for line in f if line.strip()]

selected_samples = random.sample(test_image_paths, min(3, len(test_image_paths)))

fig, axes = plt.subplots(len(selected_samples), 2, figsize=(14, 5 * len(selected_samples)))
if len(selected_samples) == 1:
    axes = np.expand_dims(axes, axis=0)

for idx, test_p in enumerate(selected_samples):
    raw = cv2.imread(str(test_p))
    gt_img = cv2.cvtColor(raw.copy(), cv2.COLOR_BGR2RGB)
    h, w, _ = gt_img.shape
    
    # Render Ground Truth
    ann_p = LABELS_DIR / f"{test_p.stem}.txt"
    if ann_p.exists():
        with open(ann_p, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cid = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
                x1, y1 = int((xc - bw/2) * w), int((yc - bh/2) * h)
                x2, y2 = int((xc + bw/2) * w), int((yc + bh/2) * h)
                cv2.rectangle(gt_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(gt_img, CLASS_NAMES.get(cid, str(cid)), (x1, max(18, y1-4)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    # Render Predictions
    pred_res = eval_model.predict(source=str(test_p), imgsz=imgsz, device=device, conf=0.25, verbose=False)[0]
    pred_img = cv2.cvtColor(pred_res.plot(), cv2.COLOR_BGR2RGB)
    
    axes[idx, 0].imshow(gt_img)
    axes[idx, 0].set_title(f"Ground Truth ({test_p.name})")
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(pred_img)
    axes[idx, 1].set_title(f"YOLO11 Prediction ({test_p.name})")
    axes[idx, 1].axis('off')

plt.tight_layout()
plt.show()

## 13. Model Export (ONNX Format)
Export trained weights to ONNX format with dynamic batching for edge deployment (such as smart sprayers, agricultural drones, or embedded robots).

In [ ]:
if best_weights.exists():
    try:
        onnx_path = eval_model.export(format="onnx", dynamic=True)
        print(f"Model successfully exported to ONNX: {onnx_path}")
    except Exception as e:
        print(f"ONNX export notification: {e}")
else:
    print("No trained best_weights found to export. Skipping ONNX export.")